In [ ]:
%run ./evaluation

In [ ]:
import mlflow
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, TimeSeriesSplit

In [ ]:
def run_gridsearch(model_class, model_type, param_grid, X_train, y_train, X_test, y_test,
                    log_model=True, run_name=None, n_iter=None, cv_splits=3):
    # TimeSeriesSplit avoids leakage
    base_estimator = model_class.build({}).estimator
    cv = TimeSeriesSplit(n_splits=cv_splits)
    n_combos = int(np.prod([len(v) for v in param_grid.values()]))

    # Sample a big space instead of exhausting it
    if n_iter and n_iter < n_combos:
        search = RandomizedSearchCV(
            base_estimator, param_grid, n_iter=n_iter, random_state=0,
            scoring="neg_mean_absolute_error", cv=cv,
        )
    else:
        search = GridSearchCV(
            base_estimator, param_grid,
            scoring="neg_mean_absolute_error", cv=cv,
        )
    search.fit(X_train, y_train)

    # Every candidate failed, sklearn scores those NaN and stays silent
    if np.isnan(search.best_score_):
        raise RuntimeError(f"{model_type}: every candidate fit failed during search")

    model = model_class(search.best_estimator_)
    model.reset_forecast()
    metrics = compute_metrics(y_test, model.predict(X_test))

    with mlflow.start_run(run_name=run_name) as run:
        mlflow.log_param("model_type", model_type)
        mlflow.log_param("search_candidates", min(n_iter or n_combos, n_combos))
        mlflow.log_param("cv_splits", cv_splits)
        for k, v in search.best_params_.items():
            mlflow.log_param(k, v)
        mlflow.log_metric("mae", metrics["mae"])
        mlflow.log_metric("rmse", metrics["rmse"])
        mlflow.log_metric("smape", metrics["smape"])
        # log_model=False saves no artifact
        if log_model:
            mlflow.sklearn.log_model(
                model.estimator, artifact_path="model",
                input_example=X_train.head(3),
                signature=mlflow.models.infer_signature(X_train, y_train),
            )
        run_id = run.info.run_id

    return run_id, metrics["mae"], metrics["rmse"], metrics["smape"], search.best_params_